# 05 — Test select-window pool12 (2023)

Evaluate a model trained in [`04_Sh_train_select_window_12.ipynb`](04_Sh_train_select_window_12.ipynb) on the **2023 test export** from [`01_export_select_window_pool12.ipynb`](01_export_select_window_pool12.ipynb).

| Item | Value |
|---|---|
| Train export | `Shalow/data/select_window_12/sample_13_22_uv_dt360` |
| Test export | `Shalow/data/select_window_12/sample_23_uv_dt360` |
| Test pool | `_sims_pool_k1.npy` — **all 12 catalog windows** per noon snapshot |
| Metrics | Per-window RMSE/L1 + overall average (single-step +15d forecast) |
| Rollout viz | **36 PNGs** — Δu,Δv increment plots (1 snapshot × 12 windows × 3 horizons) |
| Rollout metrics | Speed RMSE — persistence vs model at **+15d / +30d / +45d** (center 4×4 + full field) |
| Rollout plots | `speed_rmse_*.png` — bar, per-horizon, and curve plots |
| Per-window rollout | `by_window/window_01` … `window_12` — separate metrics + plots per catalog window |
| Cross-window compare | `by_window/heatmap_*.png` — loss & skill heatmaps (windows 1–12 vs pooled) |

**Run order:** … → per-window rollout → **cross-window heatmaps**.


In [1]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

NOTEBOOK_DIR = Path.cwd().resolve()
SHALOW_DIR = NOTEBOOK_DIR if NOTEBOOK_DIR.name == "Shalow" else NOTEBOOK_DIR / "Shalow"
ROOT = SHALOW_DIR.parent
os.chdir(SHALOW_DIR)
for _p in (ROOT, SHALOW_DIR):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from window_paths import pool_prefix, test_prefix
import importlib
import run_registry as _run_registry
importlib.reload(_run_registry)
from run_registry import (
    RUN_TAG,
    resolve_model_checkpoint,
    rollout_eval_name,
    save_rollout_results,
    run_dir,
    run_name,
    test_rollout_dir_with_model,
    update_comparison_index,
    weight_decay_run_tag,
    generator_run_tag,
    dropout_run_tag,
)
from models.build import import_unet2d
from src.train.channel_select import resolve_channel_indices, resolve_target_channel_indices
from src.train.random_window_dataset import (
    WindowPoolEvalDataset,
    evaluate_window_pool_metrics,
    load_window_pool,
    norm_stats_from_exported_sims,
)

DATA_DIR = SHALOW_DIR / "data"
MODELS_DIR = SHALOW_DIR / "Models" / "adv_no"

# --- Must match 04_Sh_train_select_window_12.ipynb ---
EXPORT_KEY = "select_window_12"
DT = 360
WEIGHT_DECAY = 1e-3
VAL_SPLIT_MODE = "random"
N_WINDOWS_PER_SAMPLE = 12
USE_WINDOW_POOL = True
DYNAMIC_RANDOM_CROP = True
TRAIN_CROPS_PER_ANCHOR = 1
TRAIN_SLOTS_PER_EPOCH = 6  # must match 04_Sh_train_select_window_12.ipynb
LOSS_MODE = "l1"
LOSS_SPATIAL_MODE = "full"
GENERATOR = "gUnet"
UNET_DROPOUT = 0.1
DEVICE = "cuda:7"

_val_suffix = "chrono" if VAL_SPLIT_MODE == "chronological" else VAL_SPLIT_MODE
_slots_tag = (
    ""
    if int(TRAIN_SLOTS_PER_EPOCH) == 1
    else f"_slots{int(TRAIN_SLOTS_PER_EPOCH)}"
)
_crop_prefix = "select_window_12"
_loss_tag = "advno" if LOSS_MODE == "adv_no" else "l1"
EXPERIMENT_TAG = f"{_crop_prefix}_{_loss_tag}{_val_suffix}{_slots_tag}"

RUN_TAG = weight_decay_run_tag(RUN_TAG, WEIGHT_DECAY)
RUN_TAG = dropout_run_tag(RUN_TAG, UNET_DROPOUT)
RUN_TAG = generator_run_tag(RUN_TAG, GENERATOR)
RUN_TAG = f"{RUN_TAG}_{EXPERIMENT_TAG}"

TRAIN_DATA_PREFIX = pool_prefix(DATA_DIR, EXPORT_KEY, dt=DT)
TEST_DATA_PREFIX = test_prefix(DATA_DIR, EXPORT_KEY, dt=DT)
RUN_NAME = run_name(EXPORT_KEY, dt=DT, run_tag=RUN_TAG)
MODEL_RUN_DIR = run_dir(MODELS_DIR, EXPORT_KEY, dt=DT, run_tag=RUN_TAG)

# Or set an explicit checkpoint dir:
# MODEL_RUN_DIR = MODELS_DIR / "shalow_adv_no_uv_13_22_select_window_12_dt360_..."

train_cfg, MODEL_PATH, STATS_PATH, BEST_EPOCH, BEST_VAL_LOSS, MODEL_RUN_NAME = (
    resolve_model_checkpoint(MODEL_RUN_DIR, expected_run_name=RUN_NAME)
)
RUN_DIR = MODEL_RUN_DIR
ROLLOUT_NAME = rollout_eval_name(MODEL_RUN_NAME, f"{EXPORT_KEY}_test2023")
TEST_ROLLOUT_DIR = test_rollout_dir_with_model(SHALOW_DIR, f"{EXPORT_KEY}_test2023", MODEL_RUN_NAME)
SAMPLES_DIR = TEST_ROLLOUT_DIR / "samples"
for _d in (TEST_ROLLOUT_DIR, SAMPLES_DIR):
    _d.mkdir(parents=True, exist_ok=True)

iD = int(train_cfg["iD"])
CHANNEL_INDICES = resolve_channel_indices(iD, train_cfg.get("channel_indices"))
TARGET_CHANNEL_INDICES = resolve_target_channel_indices(
    CHANNEL_INDICES, train_cfg.get("target_channel_indices")
)
iD_in = len(CHANNEL_INDICES)
UNET_DIM = int(train_cfg.get("unet_dim", 32))
UNET_DIM_MULTS = tuple(train_cfg.get("unet_dim_mults", (1, 2, 4, 8)))


def window_number(slot_idx: int) -> int:
    """User-facing catalog window index (1..N). Internal array slots stay 0..N-1."""
    return int(slot_idx) + 1

_split_path = MODEL_RUN_DIR / "train_val_split.json"
_train_idx = np.asarray(json.loads(_split_path.read_text())["train_indices"], dtype=np.int64)
mean_x, std_x, mean_y, std_y = norm_stats_from_exported_sims(
    TRAIN_DATA_PREFIX,
    _train_idx,
    channel_indices=train_cfg.get("channel_indices"),
    target_channel_indices=train_cfg.get("target_channel_indices"),
)

print("Working directory:", Path.cwd())
print("EXPORT_KEY:", EXPORT_KEY)
print("EXPERIMENT_TAG:", EXPERIMENT_TAG)
print("Train data:", TRAIN_DATA_PREFIX)
print("Test data:", TEST_DATA_PREFIX)
print("Model run:", MODEL_RUN_NAME)
print("Best checkpoint:", MODEL_PATH)
if BEST_EPOCH:
    print(f"  best val epoch {BEST_EPOCH}: {BEST_VAL_LOSS:.6f}")
print("Test&Rollout:", TEST_ROLLOUT_DIR)


Working directory: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow
EXPORT_KEY: select_window_12
EXPERIMENT_TAG: select_window_12_l1random_slots6
Train data: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/select_window_12/sample_13_22_uv_dt360
Test data: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/select_window_12/sample_23_uv_dt360
Model run: shalow_adv_no_uv_13_22_select_window_12_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_select_window_12_l1random_slots6
Best checkpoint: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Models/adv_no/shalow_adv_no_uv_13_22_select_window_12_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_select_window_12_l1random_slots6/best_model.pth
  best val epoch 199: 0.303154
Test&Rollout: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Test&Rollout/shalow_adv_no_uv_13_22_select_window_12_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_select_window_12_l1random_slots6__on__select_window_12_test2023


In [2]:
import json
from pathlib import Path

_pool_path = Path(f"{TEST_DATA_PREFIX}_sims_pool_k1.npy")
_meta_path = Path(f"{TEST_DATA_PREFIX}_window_pool_meta.json")
if not _pool_path.exists():
    raise FileNotFoundError(
        f"Missing test window pool: {_pool_path}\n"
        "Run the 2023 test export cell in 01_export_select_window_pool12.ipynb."
    )

import numpy as np
pool_sims = np.load(_pool_path, mmap_mode="r")
mask_pool = np.load(f"{TEST_DATA_PREFIX}_mask_pool_k1.npy", mmap_mode="r")
day = np.load(f"{TEST_DATA_PREFIX}_day.npy")
hour = np.load(f"{TEST_DATA_PREFIX}_hour.npy")
n_snap, n_win = pool_sims.shape[0], pool_sims.shape[1]
print(f"Test pool: {pool_sims.shape}  ({n_snap} snapshots × {n_win} catalog windows)")
print(f"  mask mean land: {mask_pool.mean():.2%}")
if _meta_path.exists():
    meta = json.loads(_meta_path.read_text())
    print(f"  catalog: {meta.get('catalog_id')} | window_mode: {meta.get('window_mode')}")
print(f"  eval rows (all windows): {n_snap * n_win}")


Test pool: (319, 12, 4, 2, 128, 128)  (319 snapshots × 12 catalog windows)
  mask mean land: 4.45%
  catalog: magic_window_12 | window_mode: catalog
  eval rows (all windows): 3828


## Single-step forecast — per-window RMSE (all 12 windows × all test snapshots)


In [3]:
Unet2D = import_unet2d(GENERATOR)
Par = {
    "lb": 3,
    "nf": iD,
    "nf_in": iD_in,
    "nx": int(pool_sims.shape[-2]),
    "ny": int(pool_sims.shape[-1]),
    "inp_shift": 0.0,
    "inp_scale": 1.0,
    "out_shift": 0.0,
    "out_scale": 1.0,
    "t_shift": 0.0,
    "t_scale": 1.0,
}
_unet_kwargs = {}
if GENERATOR == "gUnet":
    _unet_kwargs["dropout"] = UNET_DROPOUT
model = Unet2D(
    dim=UNET_DIM,
    Par=Par,
    dim_mults=UNET_DIM_MULTS,
    channels=Par["lb"] * iD_in,
    **_unet_kwargs,
)
model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu", weights_only=True))
model = model.to(DEVICE).eval()

window_pool = load_window_pool(TEST_DATA_PREFIX, train_crops_per_anchor=1)
test_indices = np.arange(window_pool["sims"].shape[0], dtype=np.int64)
test_day = day
test_hour = hour

test_set = WindowPoolEvalDataset(
    window_pool["sims"],
    window_pool["mask"],
    test_indices,
    test_day,
    test_hour,
    mean_x=mean_x,
    std_x=std_x,
    mean_y=mean_y,
    std_y=std_y,
    channel_indices=CHANNEL_INDICES,
    target_channel_indices=TARGET_CHANNEL_INDICES,
)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

metrics = evaluate_window_pool_metrics(
    model,
    test_loader,
    DEVICE,
    iD_in=iD_in,
    n_windows=window_pool["n_windows"],
    loss_metric="components",
)
print(f"Overall RMSE: {metrics['overall_rmse']:.6f} | Overall L1: {metrics['overall_l1']:.6f}")
print(f"Samples: {metrics['n_samples']} ({metrics['n_snapshots']} snapshots × {window_pool['n_windows']} windows)")
print("\nPer-window RMSE:")
for i in range(window_pool["n_windows"]):
    key = f"window_{i:02d}"
    l1_key = f"{key}_l1"
    print(f"  window {window_number(i):02d}: RMSE={metrics[key]:.6f}  L1={metrics.get(l1_key, float('nan')):.6f}")

results_payload = {
    "rollout_name": ROLLOUT_NAME,
    "model_run": MODEL_RUN_NAME,
    "test_prefix": TEST_DATA_PREFIX,
    "best_epoch": BEST_EPOCH,
    "best_val_loss": BEST_VAL_LOSS,
    "train_slots_per_epoch": TRAIN_SLOTS_PER_EPOCH,
    "overall_rmse": metrics["overall_rmse"],
    "overall_l1": metrics["overall_l1"],
    "per_window_metrics": metrics,
}
out_path = TEST_ROLLOUT_DIR / "test_window_metrics.json"
out_path.write_text(json.dumps(results_payload, indent=2))
print(f"\nSaved: {out_path}")

save_rollout_results(TEST_ROLLOUT_DIR, results_payload)
update_comparison_index(
    MODELS_DIR,
    run_name_value=ROLLOUT_NAME,
    export_key=f"{EXPORT_KEY}_test2023",
    run_dir_path=MODEL_RUN_DIR,
    rollout_results=results_payload,
    rollout_output_dir=TEST_ROLLOUT_DIR,
)
print("Updated comparison index")


Overall RMSE: 0.675910 | Overall L1: 0.516795
Samples: 3828 (319 snapshots × 12 windows)

Per-window RMSE:
  window 01: RMSE=0.632515  L1=0.472088
  window 02: RMSE=0.644262  L1=0.484539
  window 03: RMSE=0.869881  L1=0.665623
  window 04: RMSE=0.766244  L1=0.567407
  window 05: RMSE=0.601339  L1=0.459638
  window 06: RMSE=0.699329  L1=0.548183
  window 07: RMSE=0.498607  L1=0.390524
  window 08: RMSE=0.530409  L1=0.406065
  window 09: RMSE=0.623552  L1=0.474160
  window 10: RMSE=0.751237  L1=0.580851
  window 11: RMSE=0.835430  L1=0.642116
  window 12: RMSE=0.658112  L1=0.510343

Saved: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Test&Rollout/shalow_adv_no_uv_13_22_select_window_12_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_select_window_12_l1random_slots6__on__select_window_12_test2023/test_window_metrics.json
Updated comparison index


## Rollout viz — +15d / +30d / +45d (12 catalog windows)

3-step **autoregressive** u,v rollout at **+15d / +30d / +45d** (DT=360 h), matching [`05_Sh_evaluate_magic_window_rollout_pool12_l1random_k1.ipynb`](05_Sh_evaluate_magic_window_rollout_pool12_l1random_k1.ipynb).

- **+15d GT** from export frame 3; **+30d / +45d GT** from Zarr (catalog crop per slot)
- Saves **36 PNGs** (`sample_01_window01_snapYYYY_plus15d.png` … `window12`) for one random test snapshot (`PLOT_SEED`)
- Catalog windows numbered **1–12** (internal array slots 0–11)
- Plots normalized **Δu, Δv** increments (same layout as magic-window rollout viz)


In [4]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from Magic_window.window_catalog import get_resolved_catalog
from src.data.dataloaders.guiana_margem_tensor import GuianaDataloader
from src.export.random_window_export import crop_sequence_with_catalog_window
from src.train.fix_window_viz import (
    _uv_with_speed,
    normalized_uv_increment,
    rollout_pred_increment,
    save_gaussian_circles_sample_figure,
)

DT_STEP_DAYS = DT // 24
EVAL_HORIZONS = (DT_STEP_DAYS, 2 * DT_STEP_DAYS, 3 * DT_STEP_DAYS)
H1, H2, H3 = EVAL_HORIZONS

export_indices = np.load(f"{TEST_DATA_PREFIX}_indices.npy")
_pool_meta = json.loads(Path(f"{TEST_DATA_PREFIX}_window_pool_meta.json").read_text())
catalog = get_resolved_catalog(_pool_meta["catalog_id"])
_export_cfg = json.loads(Path(f"{TEST_DATA_PREFIX}_export_config.json").read_text())

_zarr_kwargs = dict(
    path=_export_cfg["zarr_path"],
    dt=DT,
    conditioning_steps=3,
    forecast_steps=1,
    transform=None,
)
if _export_cfg.get("start_time"):
    _zarr_kwargs["start_time"] = _export_cfg["start_time"]
if _export_cfg.get("end_time"):
    _zarr_kwargs["end_time"] = _export_cfg["end_time"]
zarr_ds = GuianaDataloader(**_zarr_kwargs)

mean_x_flat = mean_x.repeat(1, 3, 1, 1)
std_x_flat = std_x.repeat(1, 3, 1, 1)
my = mean_y.view(-1, 1, 1)
sy = std_y.view(-1, 1, 1)


def _ocean_weight(mask_t: torch.Tensor, like: torch.Tensor) -> torch.Tensor:
    return (mask_t == 0).to(dtype=like.dtype, device=like.device).unsqueeze(0).unsqueeze(0)


def _time_scalar(day_val: float, hour_val: float) -> torch.Tensor:
    return torch.tensor(
        [float(day_val) / 365.0 + float(hour_val) / 365.0 / 365.0],
        dtype=torch.float32,
        device=DEVICE,
    )


def load_catalog_uv_at_horizon(
    zarr_idx: int,
    slot: int,
    horizon_days: int,
) -> torch.Tensor | None:
    """Physical u,v at anchor + horizon for one catalog slot."""
    times = zarr_ds.sample_times(int(zarr_idx))
    anchor = pd.Timestamp(times[2])
    t = anchor + pd.Timedelta(days=int(horizon_days))
    if t not in zarr_ds.time_to_idx:
        return None
    fields = zarr_ds.load_uvssh_frame(t).astype(np.float32, copy=False)
    fields = np.nan_to_num(fields, nan=0.0)
    mask_full = (~np.isfinite(fields[:1])).astype(np.float32)[0]
    seq_c, _ = crop_sequence_with_catalog_window(
        fields[:2][np.newaxis, ...],
        mask_full,
        catalog[slot],
    )
    return torch.from_numpy(np.asarray(seq_c[0], dtype=np.float32))


def _uv_field(t: torch.Tensor) -> torch.Tensor:
    """Normalize increment tensor to [C, H, W] (GT is 3D, pred is often 4D)."""
    return t[0] if t.dim() == 4 else t


def autoregressive_rollout(
    x_norm: torch.Tensor,
    ocean: torch.Tensor,
    t_base: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """3-step normalized absolute u,v states at +15d / +30d / +45d."""
    dt_frac = DT_STEP_DAYS / 365.0
    x = x_norm
    inc1 = model(x[:, -3 * iD_in :], t_base) * ocean
    pred1 = (inc1 + x[:, -iD_in:]) * ocean

    x2 = torch.cat([x[:, iD_in : 2 * iD_in], x[:, 2 * iD_in : 3 * iD_in], pred1], dim=1) * ocean
    inc2 = model(x2[:, -3 * iD_in :], t_base + dt_frac) * ocean
    pred2 = (inc2 + x2[:, -iD_in:]) * ocean

    x3 = torch.cat([x2[:, iD_in : 2 * iD_in], x2[:, -iD_in:], pred2], dim=1) * ocean
    inc3 = model(x3[:, -3 * iD_in :], t_base + 2 * dt_frac) * ocean
    pred3 = (inc3 + pred2) * ocean
    return pred1.cpu(), pred2.cpu(), pred3.cpu()


PLOT_SEED = 42
rng = np.random.default_rng(PLOT_SEED)
snap = int(rng.integers(0, n_snap))
zarr_idx = int(export_indices[snap])
print(f"Rollout horizons: +{H1}d, +{H2}d, +{H3}d (DT={DT}h)")
print(
    f"Viz: snapshot {snap} (zarr idx {zarr_idx}) × {n_win} catalog windows × 3 horizons "
    f"→ {n_win * 3} PNGs"
)

missing_gt = {H2: 0, H3: 0}

with torch.no_grad():
    for slot in range(n_win):
        win = window_number(slot)
        seq = np.asarray(pool_sims[snap, slot], dtype=np.float32).copy()
        mask_np = np.asarray(mask_pool[snap, slot], dtype=np.float32).copy()
        mask_t = torch.from_numpy(mask_np)

        s2 = torch.from_numpy(seq[2].copy())
        s3 = torch.from_numpy(seq[3].copy())
        s4 = s3

        s5 = load_catalog_uv_at_horizon(zarr_idx, slot, H2)
        if s5 is None:
            missing_gt[H2] += 1
            s5 = torch.zeros_like(s3)
        s6 = load_catalog_uv_at_horizon(zarr_idx, slot, H3)
        if s6 is None:
            missing_gt[H3] += 1
            s6 = torch.zeros_like(s3)

        frames = [torch.from_numpy(seq[i].copy()) for i in range(3)]
        x_raw = torch.cat([f.unsqueeze(0) for f in frames], dim=1)
        x_norm = ((x_raw - mean_x_flat) / (std_x_flat + 1e-8)) * _ocean_weight(mask_t, x_raw)
        t_base = _time_scalar(day[snap], hour[snap])

        pred1, pred2, pred3 = autoregressive_rollout(
            x_norm.to(DEVICE),
            _ocean_weight(mask_t, x_norm).to(DEVICE),
            t_base,
        )

        inc_gt1 = normalized_uv_increment(s3, s2, sy, mask_t)
        inc_gt2 = normalized_uv_increment(s5, s4, sy, mask_t)
        inc_gt3 = normalized_uv_increment(s6, s5, sy, mask_t)
        inc_pred1 = rollout_pred_increment(pred1, s2, my, sy, mask_t)
        inc_pred2 = rollout_pred_increment(pred2, s4, my, sy, mask_t)
        inc_pred3 = rollout_pred_increment(pred3, s5, my, sy, mask_t)

        rollout_horizons = [
            (H1, inc_pred1, inc_gt1, "export"),
            (H2, inc_pred2, inc_gt2, "zarr"),
            (H3, inc_pred3, inc_gt3, "zarr"),
        ]

        for horizon_days, pred_inc, gt_inc, gt_src in rollout_horizons:
            pred = _uv_field(pred_inc.detach().cpu())
            target = _uv_field(gt_inc.detach().cpu())
            pred_plot, names = _uv_with_speed(pred)
            tgt_plot, _ = _uv_with_speed(target)
            out = SAMPLES_DIR / (
                f"sample_{win:02d}_window{win:02d}_snap{snap:04d}_plus{horizon_days:02d}d.png"
            )
            mae = float(torch.abs(pred[:2] - target[:2])[:, mask_t == 0].mean())
            save_gaussian_circles_sample_figure(
                pred_plot,
                tgt_plot,
                mask_t,
                out,
                channel_names=names,
                residual_metric="l1",
                title=(
                    f"window {win} | snap {snap} | +{horizon_days}d Δu,Δv (GT {gt_src}) | "
                    f"{MODEL_RUN_NAME}"
                ),
                dpi=150,
                fig_scale=2.6,
                circle_radii=(),
                inner_circle_radii=(),
            )
            print(f"Saved {out.name} | masked uv MAE={mae:.4f}")

for h_days, n_miss in missing_gt.items():
    if n_miss:
        print(f"Warning: {n_miss} catalog windows missing Zarr GT at +{h_days}d")

print(f"Done: {n_win * 3} samples in {SAMPLES_DIR}")


Rollout horizons: +15d, +30d, +45d (DT=360h)
Viz: snapshot 28 (zarr idx 684) × 12 catalog windows × 3 horizons → 36 PNGs
Saved sample_01_window01_snap0028_plus15d.png | masked uv MAE=0.4469
Saved sample_01_window01_snap0028_plus30d.png | masked uv MAE=0.4723
Saved sample_01_window01_snap0028_plus45d.png | masked uv MAE=0.4821
Saved sample_02_window02_snap0028_plus15d.png | masked uv MAE=0.4152
Saved sample_02_window02_snap0028_plus30d.png | masked uv MAE=0.4592
Saved sample_02_window02_snap0028_plus45d.png | masked uv MAE=0.6270
Saved sample_03_window03_snap0028_plus15d.png | masked uv MAE=0.5996
Saved sample_03_window03_snap0028_plus30d.png | masked uv MAE=0.4691
Saved sample_03_window03_snap0028_plus45d.png | masked uv MAE=0.6145
Saved sample_04_window04_snap0028_plus15d.png | masked uv MAE=0.6677
Saved sample_04_window04_snap0028_plus30d.png | masked uv MAE=0.5996
Saved sample_04_window04_snap0028_plus45d.png | masked uv MAE=0.6061
Saved sample_05_window05_snap0028_plus15d.png | mas

## Full test rollout inference (+15d / +30d / +45d)

3-step autoregressive rollout on **all** test rows (`n_snapshots × 12 catalog windows`). Loads +30d/+45d GT from Zarr (one read per snapshot, 12 catalog crops).

Reuses Zarr/catalog from the rollout viz cell when already run.

In [5]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from src.export.random_window_export import crop_sequence_with_catalog_window

if "EVAL_HORIZONS" not in globals():
    DT_STEP_DAYS = DT // 24
    EVAL_HORIZONS = (DT_STEP_DAYS, 2 * DT_STEP_DAYS, 3 * DT_STEP_DAYS)
    H1, H2, H3 = EVAL_HORIZONS

if "zarr_ds" not in globals():
    from Magic_window.window_catalog import get_resolved_catalog
    from src.data.dataloaders.guiana_margem_tensor import GuianaDataloader

    export_indices = np.load(f"{TEST_DATA_PREFIX}_indices.npy")
    _pool_meta = json.loads(Path(f"{TEST_DATA_PREFIX}_window_pool_meta.json").read_text())
    catalog = get_resolved_catalog(_pool_meta["catalog_id"])
    _export_cfg = json.loads(Path(f"{TEST_DATA_PREFIX}_export_config.json").read_text())
    _zarr_kwargs = dict(
        path=_export_cfg["zarr_path"],
        dt=DT,
        conditioning_steps=3,
        forecast_steps=1,
        transform=None,
    )
    if _export_cfg.get("start_time"):
        _zarr_kwargs["start_time"] = _export_cfg["start_time"]
    if _export_cfg.get("end_time"):
        _zarr_kwargs["end_time"] = _export_cfg["end_time"]
    zarr_ds = GuianaDataloader(**_zarr_kwargs)

if "mean_x_flat" not in globals():
    mean_x_flat = mean_x.repeat(1, 3, 1, 1)
    std_x_flat = std_x.repeat(1, 3, 1, 1)

H_crop, W_crop = int(pool_sims.shape[-2]), int(pool_sims.shape[-1])
y_mean_np = mean_y.detach().cpu().numpy().reshape(1, -1, 1, 1)
y_std_np = std_y.detach().cpu().numpy().reshape(1, -1, 1, 1)


def _batch_ocean(mask_b: torch.Tensor) -> torch.Tensor:
    return (mask_b == 0).to(dtype=torch.float32).unsqueeze(1)


def _batch_time(day_val: float, hour_val: float, batch_size: int) -> torch.Tensor:
    t = float(day_val) / 365.0 + float(hour_val) / 365.0 / 365.0
    return torch.full((batch_size,), t, dtype=torch.float32, device=DEVICE)


def load_catalog_uv_all_slots(zarr_idx: int, horizon_days: int) -> np.ndarray | None:
    times = zarr_ds.sample_times(int(zarr_idx))
    anchor = pd.Timestamp(times[2])
    t = anchor + pd.Timedelta(days=int(horizon_days))
    if t not in zarr_ds.time_to_idx:
        return None
    fields = zarr_ds.load_uvssh_frame(t).astype(np.float32, copy=False)
    fields = np.nan_to_num(fields, nan=0.0)
    mask_full = (~np.isfinite(fields[:1])).astype(np.float32)[0]
    slots = []
    for slot in range(n_win):
        seq_c, _ = crop_sequence_with_catalog_window(
            fields[:2][np.newaxis, ...],
            mask_full,
            catalog[slot],
        )
        slots.append(np.asarray(seq_c[0], dtype=np.float32))
    return np.stack(slots, axis=0)


def autoregressive_rollout_batch(
    x_norm: torch.Tensor,
    ocean: torch.Tensor,
    t_base: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    dt_frac = DT_STEP_DAYS / 365.0
    x = x_norm
    inc1 = model(x[:, -3 * iD_in :], t_base) * ocean
    pred1 = (inc1 + x[:, -iD_in:]) * ocean
    x2 = torch.cat([x[:, iD_in : 2 * iD_in], x[:, 2 * iD_in : 3 * iD_in], pred1], dim=1) * ocean
    inc2 = model(x2[:, -3 * iD_in :], t_base + dt_frac) * ocean
    pred2 = (inc2 + x2[:, -iD_in:]) * ocean
    x3 = torch.cat([x2[:, iD_in : 2 * iD_in], x2[:, -iD_in:], pred2], dim=1) * ocean
    inc3 = model(x3[:, -3 * iD_in :], t_base + 2 * dt_frac) * ocean
    pred3 = (inc3 + pred2) * ocean
    return pred1.cpu(), pred2.cpu(), pred3.cpu()


def _denorm_phys(pred_norm: torch.Tensor) -> np.ndarray:
    return (pred_norm.numpy() * y_std_np + y_mean_np).astype(np.float32, copy=False)


s3_rows, s4_rows, s5_rows, s6_rows = [], [], [], []
p1_rows, p2_rows, p3_rows = [], [], []
mask_rows = []
missing_h2 = missing_h3 = 0

print(f"Full rollout inference: {n_snap} snapshots × {n_win} windows = {n_snap * n_win} rows")
print(f"Horizons: +{H1}d, +{H2}d, +{H3}d (DT={DT}h)")

with torch.no_grad():
    for snap in range(n_snap):
        zarr_idx = int(export_indices[snap])
        s5_slots = load_catalog_uv_all_slots(zarr_idx, H2)
        s6_slots = load_catalog_uv_all_slots(zarr_idx, H3)
        if s5_slots is None:
            missing_h2 += n_win
            s5_slots = np.zeros((n_win, 2, H_crop, W_crop), dtype=np.float32)
        if s6_slots is None:
            missing_h3 += n_win
            s6_slots = np.zeros((n_win, 2, H_crop, W_crop), dtype=np.float32)

        seqs = np.asarray(pool_sims[snap], dtype=np.float32).copy()
        masks = np.asarray(mask_pool[snap], dtype=np.float32).copy()
        x_raw = torch.cat([torch.from_numpy(seqs[:, i].copy()) for i in range(3)], dim=1)
        ocean = _batch_ocean(torch.from_numpy(masks))
        x_norm = ((x_raw - mean_x_flat) / (std_x_flat + 1e-8)) * ocean
        t_base = _batch_time(float(day[snap]), float(hour[snap]), n_win)

        pred1, pred2, pred3 = autoregressive_rollout_batch(
            x_norm.to(DEVICE),
            ocean.to(DEVICE),
            t_base,
        )
        pred1_phys = _denorm_phys(pred1)
        pred2_phys = _denorm_phys(pred2)
        pred3_phys = _denorm_phys(pred3)

        for slot in range(n_win):
            s3_rows.append(seqs[slot, 2])
            s4_rows.append(seqs[slot, 3])
            s5_rows.append(s5_slots[slot])
            s6_rows.append(s6_slots[slot])
            p1_rows.append(pred1_phys[slot])
            p2_rows.append(pred2_phys[slot])
            p3_rows.append(pred3_phys[slot])
            mask_rows.append(masks[slot])

        if (snap + 1) % 50 == 0 or snap + 1 == n_snap:
            print(f"  {snap + 1}/{n_snap} snapshots")

if missing_h2:
    print(f"Warning: {missing_h2} rows missing Zarr GT at +{H2}d")
if missing_h3:
    print(f"Warning: {missing_h3} rows missing Zarr GT at +{H3}d")

s3 = torch.from_numpy(np.stack(s3_rows))
s4 = torch.from_numpy(np.stack(s4_rows))
s5 = torch.from_numpy(np.stack(s5_rows))
s6 = torch.from_numpy(np.stack(s6_rows))
p1 = torch.from_numpy(np.stack(p1_rows))
p2 = torch.from_numpy(np.stack(p2_rows))
p3 = torch.from_numpy(np.stack(p3_rows))
mask_all = torch.from_numpy(np.stack(mask_rows))
print("=>", s3.shape, p1.shape, s5.shape)

Full rollout inference: 319 snapshots × 12 windows = 3828 rows
Horizons: +15d, +30d, +45d (DT=360h)
  50/319 snapshots
  100/319 snapshots
  150/319 snapshots
  200/319 snapshots
  250/319 snapshots
  300/319 snapshots
  319/319 snapshots
=> torch.Size([3828, 2, 128, 128]) torch.Size([3828, 2, 128, 128]) torch.Size([3828, 2, 128, 128])


## Velocity magnitude & persistence error (+15d / +30d / +45d)

Run the full rollout inference cell above first.

- **Speed** $|\mathbf{u}| = \sqrt{u^2 + v^2}$
- **Persistence**: RMSE vs anchor speed $t[n]$
- **Model**: RMSE vs GT speed at each horizon

Reported for **center 4×4** and **full valid field** (matches `05_Sh_evaluate_magic_window_rollout_pool12_l1random_k1.ipynb`).

In [6]:
import json
import matplotlib.pyplot as plt
import numpy as np
import torch
from datetime import datetime, timezone

SPEED_PATCH_SIZE = 4


def _center_slices(h: int, w: int, patch: int) -> tuple[slice, slice]:
    cy, cx = h // 2, w // 2
    half = patch // 2
    if patch % 2 == 0:
        return slice(cy - half, cy + half), slice(cx - half, cx + half)
    return slice(cy - half, cy + half + 1), slice(cx - half, cx + half + 1)


def _crop_metric_region(field: np.ndarray, mask: np.ndarray, patch_size: int | None):
    if patch_size is None:
        return field, mask
    sy, sx = _center_slices(field.shape[-2], field.shape[-1], patch_size)
    return field[..., sy, sx], mask[..., sy, sx]


def speed_magnitude(states) -> np.ndarray:
    arr = states.numpy() if torch.is_tensor(states) else np.asarray(states)
    return np.sqrt(arr[:, 0] ** 2 + arr[:, 1] ** 2)


def masked_rmse(field_a: np.ndarray, field_b: np.ndarray, mask: np.ndarray, patch_size: int | None = None) -> float:
    field_a, mask = _crop_metric_region(field_a, mask, patch_size)
    field_b, _ = _crop_metric_region(field_b, mask, patch_size)
    diff2 = (field_b - field_a) ** 2
    valid = mask == 0
    if not np.any(valid):
        return float("nan")
    return float(np.sqrt(diff2[valid].mean()))


def rollout_speed_metrics(
    gt_now,
    gt_h1,
    gt_h2,
    gt_h3,
    pred_h1,
    pred_h2,
    pred_h3,
    mask,
    horizons,
    patch_size: int | None = None,
):
    speed_now = speed_magnitude(gt_now)
    gt_speeds = [speed_magnitude(gt_h1), speed_magnitude(gt_h2), speed_magnitude(gt_h3)]
    pred_speeds = [speed_magnitude(pred_h1), speed_magnitude(pred_h2), speed_magnitude(pred_h3)]
    mask_np = mask.numpy() if torch.is_tensor(mask) else np.asarray(mask)

    rows = []
    for days, gt_spd, pred_spd in zip(horizons, gt_speeds, pred_speeds):
        persist = masked_rmse(speed_now, gt_spd, mask_np, patch_size)
        model_rmse = masked_rmse(pred_spd, gt_spd, mask_np, patch_size)
        skill = 1.0 - model_rmse / persist if persist > 0 else float("nan")
        rows.append(
            {
                "horizon_days": days,
                "persistence_rmse": persist,
                "model_rmse": model_rmse,
                "skill_score": skill,
            }
        )
    return rows


def print_metrics_table(metrics, region_label: str):
    print(f"\nSpeed RMSE (|U,V|) — {region_label}")
    print(f"{'Horizon':>10} {'Persistence':>14} {'Model':>14} {'Skill':>10}")
    print("-" * 52)
    for row in metrics:
        print(
            f"+{row['horizon_days']:>2}d"
            f" {row['persistence_rmse']:14.6f}"
            f" {row['model_rmse']:14.6f}"
            f" {row['skill_score']:10.3f}"
        )


metrics_center = rollout_speed_metrics(
    s3, s4, s5, s6, p1, p2, p3, mask_all, horizons=EVAL_HORIZONS, patch_size=SPEED_PATCH_SIZE
)
metrics_full = rollout_speed_metrics(
    s3, s4, s5, s6, p1, p2, p3, mask_all, horizons=EVAL_HORIZONS, patch_size=None
)

print(f"Dataset: {TEST_DATA_PREFIX} | N={s3.shape[0]} rows | run: {ROLLOUT_NAME}")
print_metrics_table(metrics_center, f"center {SPEED_PATCH_SIZE}x{SPEED_PATCH_SIZE}")
print_metrics_table(metrics_full, "full valid field")

persist_center = [row["persistence_rmse"] for row in metrics_center]
model_center = [row["model_rmse"] for row in metrics_center]
skill_center = [row["skill_score"] for row in metrics_center]
persist_full = [row["persistence_rmse"] for row in metrics_full]
model_full = [row["model_rmse"] for row in metrics_full]
skill_full = [row["skill_score"] for row in metrics_full]

rollout_results = {
    "export_key": f"{EXPORT_KEY}_test2023",
    "rollout_name": ROLLOUT_NAME,
    "model_run": MODEL_RUN_NAME,
    "model_run_dir": str(MODEL_RUN_DIR),
    "rollout_output_dir": str(TEST_ROLLOUT_DIR),
    "eval_data_prefix": TEST_DATA_PREFIX,
    "n_test": int(s3.shape[0]),
    "n_snapshots": int(n_snap),
    "n_windows": int(n_win),
    "best_epoch": BEST_EPOCH,
    "best_val_loss": BEST_VAL_LOSS,
    "eval_horizons_days": list(EVAL_HORIZONS),
    "speed_patch_size": SPEED_PATCH_SIZE,
    "metrics_center": metrics_center,
    "metrics_full": metrics_full,
    "evaluated_at": datetime.now(timezone.utc).isoformat(),
}
results_path = save_rollout_results(TEST_ROLLOUT_DIR, rollout_results)
index_path = update_comparison_index(
    MODELS_DIR,
    run_name_value=ROLLOUT_NAME,
    export_key=f"{EXPORT_KEY}_test2023",
    run_dir_path=MODEL_RUN_DIR,
    rollout_results=rollout_results,
    rollout_output_dir=TEST_ROLLOUT_DIR,
)
print(f"Saved rollout results: {results_path}")
print(f"Updated comparison index: {index_path}")

Dataset: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/select_window_12/sample_23_uv_dt360 | N=3828 rows | run: shalow_adv_no_uv_13_22_select_window_12_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_select_window_12_l1random_slots6__on__select_window_12_test2023

Speed RMSE (|U,V|) — center 4x4
   Horizon    Persistence          Model      Skill
----------------------------------------------------
+15d       0.314790       0.242577      0.229
+30d       0.376205       0.298039      0.208
+45d       0.399705       0.331450      0.171

Speed RMSE (|U,V|) — full valid field
   Horizon    Persistence          Model      Skill
----------------------------------------------------
+15d       0.270850       0.218541      0.193
+30d       0.317676       0.261118      0.178
+45d       0.332792       0.282915      0.150
Saved rollout results: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Test&Rollout/shalow_adv_no_uv_13_22_select_window_12_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_select_

In [7]:
# Bar plots: center 4x4 and full field
labels = [f"+{d}d" for d in EVAL_HORIZONS]
x = np.arange(len(labels))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, persist, model, skill, title in [
    (axes[0], persist_center, model_center, skill_center, f"center {SPEED_PATCH_SIZE}x{SPEED_PATCH_SIZE}"),
    (axes[1], persist_full, model_full, skill_full, "full valid field"),
]:
    ax.bar(x - width / 2, persist, width, label="Persistence", color="#ff7f0e")
    ax.bar(x + width / 2, model, width, label="Model", color="#1f77b4")
    ax.set_xticks(x, labels)
    ax.set_ylabel("Speed RMSE (|U,V|)")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)
    for i, (p, m, s) in enumerate(zip(persist, model, skill)):
        ax.text(i - width / 2, p, f"{p:.3f}", ha="center", va="bottom", fontsize=8)
        ax.text(i + width / 2, m, f"{m:.3f}", ha="center", va="bottom", fontsize=8)
        ax.text(i, max(p, m) * 1.02, f"skill={s:.2f}", ha="center", va="bottom", fontsize=8)

fig.suptitle(f"Rollout speed error — {ROLLOUT_NAME}", y=1.02)
plt.tight_layout()
out_path = TEST_ROLLOUT_DIR / "speed_rmse_center_and_full.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

Saved: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Test&Rollout/shalow_adv_no_uv_13_22_select_window_12_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_select_window_12_l1random_slots6__on__select_window_12_test2023/speed_rmse_center_and_full.png


In [8]:
# Per-horizon plots: persistence vs model RMSE (+15d / +30d / +45d)
# Row 0 = center 4×4, row 1 = full valid field; one panel per horizon
width = 0.55

fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey="row")
region_rows = [
    (persist_center, model_center, skill_center, f"center {SPEED_PATCH_SIZE}×{SPEED_PATCH_SIZE}"),
    (persist_full, model_full, skill_full, "full valid field"),
]

for row, (persist, model, skill, row_title) in enumerate(region_rows):
    for col, (days, p, m, s) in enumerate(zip(EVAL_HORIZONS, persist, model, skill)):
        ax = axes[row, col]
        bars = ax.bar(["Persistence", "Model"], [p, m], color=["#ff7f0e", "#1f77b4"], width=width)
        ax.set_title(f"{row_title} | +{days}d")
        ax.set_ylabel("Speed RMSE (|U,V|)" if col == 0 else "")
        ax.grid(True, axis="y", alpha=0.3)
        for bar, val in zip(bars, [p, m]):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                val,
                f"{val:.4f}",
                ha="center",
                va="bottom",
                fontsize=9,
            )
        ax.text(0.5, 0.95, f"skill={s:.3f}", ha="center", va="top", fontsize=9, transform=ax.transAxes)

fig.suptitle(f"Persistence vs model speed RMSE by horizon — {ROLLOUT_NAME}", y=1.02, fontsize=13)
plt.tight_layout()
out_path = TEST_ROLLOUT_DIR / "speed_rmse_by_horizon_center_and_full.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

Saved: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Test&Rollout/shalow_adv_no_uv_13_22_select_window_12_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_select_window_12_l1random_slots6__on__select_window_12_test2023/speed_rmse_by_horizon_center_and_full.png


In [9]:
# Line + point curves: persistence vs model RMSE vs horizon (+15d / +30d / +45d)
horizons = np.array(EVAL_HORIZONS, dtype=float)
xtick_labels = [f"+{int(d)}d" for d in horizons]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, persist, model, title in [
    (axes[0], persist_center, model_center, f"center {SPEED_PATCH_SIZE}×{SPEED_PATCH_SIZE}"),
    (axes[1], persist_full, model_full, "full valid field"),
]:
    ax.plot(
        horizons,
        persist,
        "o-",
        color="#ff7f0e",
        linewidth=2,
        markersize=8,
        label="Persistence",
    )
    ax.plot(
        horizons,
        model,
        "s-",
        color="#1f77b4",
        linewidth=2,
        markersize=8,
        label="Model",
    )
    for d, p, m in zip(horizons, persist, model):
        ax.annotate(f"{p:.3f}", (d, p), textcoords="offset points", xytext=(0, 10), ha="center", fontsize=8, color="#cc6600")
        ax.annotate(f"{m:.3f}", (d, m), textcoords="offset points", xytext=(0, -14), ha="center", fontsize=8, color="#1a5a9a")

    ax.set_xticks(horizons)
    ax.set_xticklabels(xtick_labels)
    ax.set_xlabel("Forecast horizon")
    ax.set_ylabel("Speed RMSE (|U,V|)")
    ax.set_title(title)
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3)

fig.suptitle(f"Speed RMSE curves — {ROLLOUT_NAME}", y=1.02, fontsize=13)
plt.tight_layout()
out_path = TEST_ROLLOUT_DIR / "speed_rmse_curves_center_and_full.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

Saved: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Test&Rollout/shalow_adv_no_uv_13_22_select_window_12_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_select_window_12_l1random_slots6__on__select_window_12_test2023/speed_rmse_curves_center_and_full.png


## Per-window rollout (windows 1–12)

Runs rollout metrics **separately for each catalog window** (all `n_snap` test snapshots per window). Saves under `Test&Rollout/.../by_window/window_01/` … `window_12/`:

- `rollout_results.json` — speed RMSE (center 4×4 + full field) at +15d / +30d / +45d
- `speed_rmse_center_and_full.png`, `speed_rmse_by_horizon_center_and_full.png`, `speed_rmse_curves_center_and_full.png`

Uses tensors from the full rollout inference cell when available; otherwise runs inference per window.

In [10]:
import json
import matplotlib.pyplot as plt
import numpy as np
import torch
from datetime import datetime, timezone
from pathlib import Path

assert "rollout_speed_metrics" in globals(), "Run the speed RMSE metrics cell above first."

BY_WINDOW_DIR = TEST_ROLLOUT_DIR / "by_window"
BY_WINDOW_DIR.mkdir(parents=True, exist_ok=True)
H_crop = int(globals().get("H_crop", pool_sims.shape[-2]))
W_crop = int(globals().get("W_crop", pool_sims.shape[-1]))

if "load_catalog_uv_at_horizon" not in globals() and "load_catalog_uv_all_slots" in globals():
    def load_catalog_uv_at_horizon(zarr_idx: int, slot: int, horizon_days: int):
        arr = load_catalog_uv_all_slots(zarr_idx, horizon_days)
        if arr is None:
            return None
        return torch.from_numpy(arr[slot])

_catalog_windows = []
if "_pool_meta" in globals():
    _catalog_windows = _pool_meta.get("catalog", {}).get("windows", [])
elif Path(f"{TEST_DATA_PREFIX}_window_pool_meta.json").exists():
    _catalog_windows = json.loads(
        Path(f"{TEST_DATA_PREFIX}_window_pool_meta.json").read_text()
    ).get("catalog", {}).get("windows", [])


def _window_row_indices(slot_idx: int) -> np.ndarray:
    return np.arange(int(slot_idx), n_snap * n_win, n_win, dtype=np.int64)


def _rollout_inference_one_window(slot: int) -> tuple[torch.Tensor, ...]:
    """Rollout all snapshots for a single catalog slot (batch=1 per snap)."""
    s3_rows, s4_rows, s5_rows, s6_rows = [], [], [], []
    p1_rows, p2_rows, p3_rows = [], [], []
    mask_rows = []
    with torch.no_grad():
        for snap in range(n_snap):
            zarr_idx = int(export_indices[snap])
            seq = np.asarray(pool_sims[snap, slot], dtype=np.float32).copy()
            masks = np.asarray(mask_pool[snap, slot], dtype=np.float32).copy()
            s5 = load_catalog_uv_at_horizon(zarr_idx, slot, H2)
            s6 = load_catalog_uv_at_horizon(zarr_idx, slot, H3)
            if s5 is None:
                s5 = torch.zeros(2, H_crop, W_crop)
            if s6 is None:
                s6 = torch.zeros(2, H_crop, W_crop)

            x_raw = torch.cat(
                [torch.from_numpy(seq[i].copy()).unsqueeze(0) for i in range(3)],
                dim=1,
            )
            ocean = _batch_ocean(torch.from_numpy(masks)).to(DEVICE)
            x_norm = ((x_raw - mean_x_flat) / (std_x_flat + 1e-8)).to(DEVICE) * ocean
            t_base = _batch_time(float(day[snap]), float(hour[snap]), 1)

            pred1, pred2, pred3 = autoregressive_rollout_batch(x_norm, ocean, t_base)
            pred1_phys = _denorm_phys(pred1)[0]
            pred2_phys = _denorm_phys(pred2)[0]
            pred3_phys = _denorm_phys(pred3)[0]

            s3_rows.append(seq[2])
            s4_rows.append(seq[3])
            s5_rows.append(s5.numpy() if torch.is_tensor(s5) else s5)
            s6_rows.append(s6.numpy() if torch.is_tensor(s6) else s6)
            p1_rows.append(pred1_phys)
            p2_rows.append(pred2_phys)
            p3_rows.append(pred3_phys)
            mask_rows.append(masks)

    return tuple(
        torch.from_numpy(np.stack(rows))
        for rows in (s3_rows, s4_rows, s5_rows, s6_rows, p1_rows, p2_rows, p3_rows, mask_rows)
    )


def _save_window_speed_plots(
    win_dir: Path,
    *,
    win: int,
    catalog_key: str | None,
    persist_center,
    model_center,
    skill_center,
    persist_full,
    model_full,
    skill_full,
):
    labels = [f"+{d}d" for d in EVAL_HORIZONS]
    x = np.arange(len(labels))
    bar_w = 0.35
    win_title = f"window {win}" + (f" ({catalog_key})" if catalog_key else "")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    for ax, persist, model, skill, title in [
        (axes[0], persist_center, model_center, skill_center, f"center {SPEED_PATCH_SIZE}x{SPEED_PATCH_SIZE}"),
        (axes[1], persist_full, model_full, skill_full, "full valid field"),
    ]:
        ax.bar(x - bar_w / 2, persist, bar_w, label="Persistence", color="#ff7f0e")
        ax.bar(x + bar_w / 2, model, bar_w, label="Model", color="#1f77b4")
        ax.set_xticks(x, labels)
        ax.set_ylabel("Speed RMSE (|U,V|)")
        ax.set_title(title)
        ax.legend()
        ax.grid(True, axis="y", alpha=0.3)
    fig.suptitle(f"Rollout speed error — {win_title}", y=1.02)
    plt.tight_layout()
    p1 = win_dir / "speed_rmse_center_and_full.png"
    plt.savefig(p1, dpi=150, bbox_inches="tight")
    plt.close(fig)

    fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey="row")
    for row, (persist, model, skill, row_title) in enumerate(
        [
            (persist_center, model_center, skill_center, f"center {SPEED_PATCH_SIZE}×{SPEED_PATCH_SIZE}"),
            (persist_full, model_full, skill_full, "full valid field"),
        ]
    ):
        for col, (days, p, m, s) in enumerate(zip(EVAL_HORIZONS, persist, model, skill)):
            ax = axes[row, col]
            bars = ax.bar(["Persistence", "Model"], [p, m], color=["#ff7f0e", "#1f77b4"], width=0.55)
            ax.set_title(f"{row_title} | +{days}d")
            ax.set_ylabel("Speed RMSE (|U,V|)" if col == 0 else "")
            ax.grid(True, axis="y", alpha=0.3)
            for bar, val in zip(bars, [p, m]):
                ax.text(bar.get_x() + bar.get_width() / 2, val, f"{val:.4f}", ha="center", va="bottom", fontsize=9)
            ax.text(0.5, 0.95, f"skill={s:.3f}", ha="center", va="top", fontsize=9, transform=ax.transAxes)
    fig.suptitle(f"Persistence vs model — {win_title}", y=1.02, fontsize=13)
    plt.tight_layout()
    p2 = win_dir / "speed_rmse_by_horizon_center_and_full.png"
    plt.savefig(p2, dpi=150, bbox_inches="tight")
    plt.close(fig)

    horizons = np.array(EVAL_HORIZONS, dtype=float)
    xtick_labels = [f"+{int(d)}d" for d in horizons]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
    for ax, persist, model, title in [
        (axes[0], persist_center, model_center, f"center {SPEED_PATCH_SIZE}×{SPEED_PATCH_SIZE}"),
        (axes[1], persist_full, model_full, "full valid field"),
    ]:
        ax.plot(horizons, persist, "o-", color="#ff7f0e", linewidth=2, markersize=8, label="Persistence")
        ax.plot(horizons, model, "s-", color="#1f77b4", linewidth=2, markersize=8, label="Model")
        ax.set_xticks(horizons)
        ax.set_xticklabels(xtick_labels)
        ax.set_xlabel("Forecast horizon")
        ax.set_ylabel("Speed RMSE (|U,V|)")
        ax.set_title(title)
        ax.legend(loc="best")
        ax.grid(True, alpha=0.3)
    fig.suptitle(f"Speed RMSE curves — {win_title}", y=1.02, fontsize=13)
    plt.tight_layout()
    p3 = win_dir / "speed_rmse_curves_center_and_full.png"
    plt.savefig(p3, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return p1, p2, p3


_have_pooled = (
    "s3" in globals()
    and int(s3.shape[0]) == n_snap * n_win
    and "p1" in globals()
    and "mask_all" in globals()
)

per_window_summary = []
print(f"Per-window rollout → {BY_WINDOW_DIR}")
print(f"Source: {'pooled full inference' if _have_pooled else 'per-window inference (re-run)'}")

for slot in range(n_win):
    win = window_number(slot)
    win_dir = BY_WINDOW_DIR / f"window_{win:02d}"
    win_dir.mkdir(parents=True, exist_ok=True)
    catalog_key = None
    if slot < len(_catalog_windows):
        catalog_key = _catalog_windows[slot].get("export_key")

    if _have_pooled:
        idx = _window_row_indices(slot)
        s3_w, s4_w, s5_w, s6_w = s3[idx], s4[idx], s5[idx], s6[idx]
        p1_w, p2_w, p3_w = p1[idx], p2[idx], p3[idx]
        mask_w = mask_all[idx]
    else:
        for _name in (
            "autoregressive_rollout_batch",
            "load_catalog_uv_at_horizon",
            "_batch_ocean",
            "_batch_time",
            "_denorm_phys",
            "export_indices",
            "H_crop",
            "W_crop",
        ):
            if _name not in globals():
                raise RuntimeError(
                    f"Missing {_name!r}. Run the full rollout inference cell first, "
                    "or re-run rollout viz + inference setup."
                )
        s3_w, s4_w, s5_w, s6_w, p1_w, p2_w, p3_w, mask_w = _rollout_inference_one_window(slot)

    mc = rollout_speed_metrics(
        s3_w, s4_w, s5_w, s6_w, p1_w, p2_w, p3_w, mask_w,
        horizons=EVAL_HORIZONS, patch_size=SPEED_PATCH_SIZE,
    )
    mf = rollout_speed_metrics(
        s3_w, s4_w, s5_w, s6_w, p1_w, p2_w, p3_w, mask_w,
        horizons=EVAL_HORIZONS, patch_size=None,
    )
    pc = [r["persistence_rmse"] for r in mc]
    pm = [r["model_rmse"] for r in mc]
    sc = [r["skill_score"] for r in mc]
    pf = [r["persistence_rmse"] for r in mf]
    pmod = [r["model_rmse"] for r in mf]
    sf = [r["skill_score"] for r in mf]

    win_results = {
        "window_number": win,
        "slot_index": slot,
        "catalog_export_key": catalog_key,
        "rollout_name": ROLLOUT_NAME,
        "model_run": MODEL_RUN_NAME,
        "n_snapshots": int(n_snap),
        "eval_horizons_days": list(EVAL_HORIZONS),
        "speed_patch_size": SPEED_PATCH_SIZE,
        "metrics_center": mc,
        "metrics_full": mf,
        "evaluated_at": datetime.now(timezone.utc).isoformat(),
    }
    results_path = save_rollout_results(win_dir, win_results)
    plot_paths = _save_window_speed_plots(
        win_dir,
        win=win,
        catalog_key=catalog_key,
        persist_center=pc,
        model_center=pm,
        skill_center=sc,
        persist_full=pf,
        model_full=pmod,
        skill_full=sf,
    )
    per_window_summary.append(
        {
            "window_number": win,
            "slot_index": slot,
            "catalog_export_key": catalog_key,
            "output_dir": str(win_dir),
            "rollout_results": str(results_path),
            "plots": [str(p) for p in plot_paths],
            "metrics_center": mc,
            "metrics_full": mf,
        }
    )
    print(
        f"  window {win:02d}: N={s3_w.shape[0]} | "
        f"+{H1}d model={pm[0]:.4f} | +{H2}d model={pm[1]:.4f} | +{H3}d model={pm[2]:.4f} | "
        f"saved {win_dir.name}/"
    )

summary_path = BY_WINDOW_DIR / "per_window_rollout_summary.json"
summary_path.write_text(json.dumps(per_window_summary, indent=2))
print(f"\nSaved summary: {summary_path}")

Per-window rollout → /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Test&Rollout/shalow_adv_no_uv_13_22_select_window_12_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_select_window_12_l1random_slots6__on__select_window_12_test2023/by_window
Source: pooled full inference
  window 01: N=319 | +15d model=0.2266 | +30d model=0.2706 | +45d model=0.3112 | saved window_01/
  window 02: N=319 | +15d model=0.2259 | +30d model=0.2780 | +45d model=0.3064 | saved window_02/
  window 03: N=319 | +15d model=0.3102 | +30d model=0.4380 | +45d model=0.5072 | saved window_03/
  window 04: N=319 | +15d model=0.4044 | +30d model=0.4803 | +45d model=0.5736 | saved window_04/
  window 05: N=319 | +15d model=0.2619 | +30d model=0.3511 | +45d model=0.3411 | saved window_05/
  window 06: N=319 | +15d model=0.1957 | +30d model=0.2690 | +45d model=0.3189 | saved window_06/
  window 07: N=319 | +15d model=0.1359 | +30d model=0.1509 | +45d model=0.1590 | saved window_07/
  window 08: N=319 | +15d model=0.1121 | +30

## Cross-window comparison — loss & skill heatmaps

Compare **all 12 catalog windows** with the **pooled** rollout (all windows combined). Heatmaps: rows = windows 1–12 + pooled, columns = +15d / +30d / +45d.

- **Model speed RMSE** (loss) — center 4×4 and full field
- **Skill score** (1 − model/persistence) — center and full

Requires the per-window rollout cell above (`by_window/per_window_rollout_summary.json`) and pooled `rollout_results.json`.

In [11]:
import json

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

BY_WINDOW_DIR = TEST_ROLLOUT_DIR / "by_window"
summary_path = BY_WINDOW_DIR / "per_window_rollout_summary.json"
pooled_path = TEST_ROLLOUT_DIR / "rollout_results.json"

if "per_window_summary" in globals() and per_window_summary:
    window_entries = sorted(per_window_summary, key=lambda e: e["window_number"])
elif summary_path.exists():
    window_entries = sorted(
        json.loads(summary_path.read_text()),
        key=lambda e: e["window_number"],
    )
else:
    raise FileNotFoundError(
        f"Missing {summary_path}. Run the per-window rollout cell first."
    )

if pooled_path.exists():
    pooled_results = json.loads(pooled_path.read_text())
elif "rollout_results" in globals():
    pooled_results = rollout_results
else:
    raise FileNotFoundError(
        f"Missing {pooled_path}. Run the pooled speed RMSE metrics cell first."
    )

horizons = list(pooled_results.get("eval_horizons_days", EVAL_HORIZONS))
horizon_labels = [f"+{int(d)}d" for d in horizons]
patch_sz = int(pooled_results.get("speed_patch_size", SPEED_PATCH_SIZE))


def _short_catalog_key(export_key: str | None) -> str:
    if not export_key:
        return ""
    return export_key.replace("magicwin_", "").replace("select_window_", "")


def _row_label(win: int | str, catalog_key: str | None = None) -> str:
    if win == "all":
        return "All\n(pooled)"
    short = _short_catalog_key(catalog_key)
    return f"W{int(win):02d}" + (f"\n{short}" if short else "")


def _build_matrix(
    entries: list[dict],
    *,
    region: str,
    metric: str,
    pooled_row: list[dict],
) -> tuple[np.ndarray, list[str]]:
    """Rows = windows 1..12 then pooled; cols = horizons."""
    rows = []
    labels = []
    for entry in entries:
        metrics = entry[region]
        rows.append([float(m[metric]) for m in metrics])
        labels.append(_row_label(entry["window_number"], entry.get("catalog_export_key")))
    rows.append([float(m[metric]) for m in pooled_row])
    labels.append(_row_label("all"))
    return np.asarray(rows, dtype=np.float64), labels


def _plot_heatmap(
    ax,
    data: np.ndarray,
    row_labels: list[str],
    *,
    title: str,
    cbar_label: str,
    cmap: str,
    vmin=None,
    vmax=None,
    fmt: str = ".3f",
    highlight_last_row: bool = True,
):
    im = ax.imshow(data, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(np.arange(len(horizon_labels)))
    ax.set_xticklabels(horizon_labels)
    ax.set_yticks(np.arange(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=8)
    ax.set_title(title)
    ax.set_xlabel("Forecast horizon")
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(cbar_label, rotation=270, labelpad=12)

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            val = data[i, j]
            txt = "nan" if np.isnan(val) else format(val, fmt)
            ax.text(
                j,
                i,
                txt,
                ha="center",
                va="center",
                color="white" if val > np.nanmean(data) else "black",
                fontsize=7,
            )
    if highlight_last_row and data.shape[0] > 1:
        ax.axhline(data.shape[0] - 1.5, color="white", linewidth=2.5)
        ax.axhline(data.shape[0] - 1.5, color="black", linewidth=1.0, linestyle="--")


# --- matrices: per-window + pooled row ---
loss_center, row_labels = _build_matrix(
    window_entries,
    region="metrics_center",
    metric="model_rmse",
    pooled_row=pooled_results["metrics_center"],
)
loss_full, _ = _build_matrix(
    window_entries,
    region="metrics_full",
    metric="model_rmse",
    pooled_row=pooled_results["metrics_full"],
)
skill_center, _ = _build_matrix(
    window_entries,
    region="metrics_center",
    metric="skill_score",
    pooled_row=pooled_results["metrics_center"],
)
skill_full, _ = _build_matrix(
    window_entries,
    region="metrics_full",
    metric="skill_score",
    pooled_row=pooled_results["metrics_full"],
)

# Shared color scales (per metric type) so windows are comparable
loss_vmax = np.nanmax([loss_center, loss_full])
skill_vmin = np.nanmin([skill_center, skill_full])
skill_vmax = np.nanmax([skill_center, skill_full])

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
_plot_heatmap(
    axes[0, 0],
    loss_center,
    row_labels,
    title=f"Model speed RMSE — center {patch_sz}×{patch_sz}",
    cbar_label="RMSE (|U,V|)",
    cmap="YlOrRd",
    vmin=0.0,
    vmax=loss_vmax,
)
_plot_heatmap(
    axes[0, 1],
    loss_full,
    row_labels,
    title="Model speed RMSE — full valid field",
    cbar_label="RMSE (|U,V|)",
    cmap="YlOrRd",
    vmin=0.0,
    vmax=loss_vmax,
)
_plot_heatmap(
    axes[1, 0],
    skill_center,
    row_labels,
    title=f"Skill score — center {patch_sz}×{patch_sz}",
    cbar_label="Skill (1 − model/persist)",
    cmap="RdYlGn",
    vmin=skill_vmin,
    vmax=skill_vmax,
)
_plot_heatmap(
    axes[1, 1],
    skill_full,
    row_labels,
    title="Skill score — full valid field",
    cbar_label="Skill (1 − model/persist)",
    cmap="RdYlGn",
    vmin=skill_vmin,
    vmax=skill_vmax,
)
fig.suptitle(
    f"Per-window vs pooled rollout — {ROLLOUT_NAME}\n"
    f"(rows W01–W12 = single window; bottom row = all windows pooled)",
    y=1.02,
    fontsize=12,
)
plt.tight_layout()
compare_path = BY_WINDOW_DIR / "heatmap_loss_and_skill_center_and_full.png"
plt.savefig(compare_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {compare_path}")

# --- side-by-side: model RMSE only (larger) ---
fig2, axes2 = plt.subplots(1, 2, figsize=(13, 8))
_plot_heatmap(
    axes2[0],
    loss_center,
    row_labels,
    title=f"Loss (model RMSE) — center {patch_sz}×{patch_sz}",
    cbar_label="RMSE",
    cmap="YlOrRd",
    vmin=0.0,
    vmax=loss_vmax,
)
_plot_heatmap(
    axes2[1],
    loss_full,
    row_labels,
    title="Loss (model RMSE) — full field",
    cbar_label="RMSE",
    cmap="YlOrRd",
    vmin=0.0,
    vmax=loss_vmax,
)
fig2.suptitle(f"Model speed RMSE heatmap — windows 1–12 vs pooled", y=1.02)
plt.tight_layout()
loss_path = BY_WINDOW_DIR / "heatmap_model_rmse_by_window.png"
plt.savefig(loss_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {loss_path}")

fig3, axes3 = plt.subplots(1, 2, figsize=(13, 8))
_plot_heatmap(
    axes3[0],
    skill_center,
    row_labels,
    title=f"Skill — center {patch_sz}×{patch_sz}",
    cbar_label="Skill",
    cmap="RdYlGn",
    vmin=skill_vmin,
    vmax=skill_vmax,
)
_plot_heatmap(
    axes3[1],
    skill_full,
    row_labels,
    title="Skill — full field",
    cbar_label="Skill",
    cmap="RdYlGn",
    vmin=skill_vmin,
    vmax=skill_vmax,
)
fig3.suptitle(f"Skill score heatmap — windows 1–12 vs pooled", y=1.02)
plt.tight_layout()
skill_path = BY_WINDOW_DIR / "heatmap_skill_by_window.png"
plt.savefig(skill_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {skill_path}")

# --- numeric table: per-window minus pooled (model RMSE, center) ---
pooled_loss_center = loss_center[-1]
delta_center = loss_center[:-1] - pooled_loss_center[np.newaxis, :]
print("\nΔ model RMSE vs pooled (center) — negative = better than pooled:")
header = " " * 8 + "  ".join(f"{h:>8}" for h in horizon_labels)
print(header)
for i, entry in enumerate(window_entries):
    vals = "  ".join(f"{v:8.4f}" for v in delta_center[i])
    print(f"W{entry['window_number']:02d}      {vals}")

Saved: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Test&Rollout/shalow_adv_no_uv_13_22_select_window_12_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_select_window_12_l1random_slots6__on__select_window_12_test2023/by_window/heatmap_loss_and_skill_center_and_full.png
Saved: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Test&Rollout/shalow_adv_no_uv_13_22_select_window_12_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_select_window_12_l1random_slots6__on__select_window_12_test2023/by_window/heatmap_model_rmse_by_window.png
Saved: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/Test&Rollout/shalow_adv_no_uv_13_22_select_window_12_dt360_vgg_warm10_adv60_wd1e-3_drop0.1_select_window_12_l1random_slots6__on__select_window_12_test2023/by_window/heatmap_skill_by_window.png

Δ model RMSE vs pooled (center) — negative = better than pooled:
            +15d      +30d      +45d
W01       -0.0159   -0.0274   -0.0202
W02       -0.0167   -0.0201   -0.0250
W03        0.0676    0.1400    0.1758
W